In [5]:
cd /Users/karolinegriesbach/Documents/Innkeepr/Git/evaluation-and-execution-scripts/

In [6]:
import json
import logging
import pandas as pd
from datetime import datetime, timedelta

from general_functions.conncet_s3 import S3Connection
from general_functions.return_workspace_ids import return_workspace_ids
from general_functions.call_api_with_account_id import call_api_with_accountId
from general_functions.constants import return_api_url
from general_functions.sanitize_accout_name import sanitize_account_name


In [7]:
workspaces = return_workspace_ids()
url = return_api_url()
date = (datetime.now() - timedelta(days=30)).strftime("%Y-%m-%d")
s3 = S3Connection()
print(date)
print(url)
check_workspaces = []
add_scaling_file = []

In [8]:
for workspace in workspaces:
    print(workspace)
    if workspace["name"] in check_workspaces:
        continue
    sanitized_workspace = sanitize_account_name(workspace["name"])
    models = call_api_with_accountId(
                endpoint_url=f"{url}/models/query",
                accountID=workspace["id"],
                content={"ceated": {"$gte": date}},
                logger=logging)
    models = pd.json_normalize(models)
    if models.empty:
        continue
    max_conversion = models[models["type"] == "conversion"]["created"].max()
    max_conversion = models[(models["type"] == "conversion") & (models["created"] == max_conversion)]
    max_causal = models[models["type"] == "causal"].groupby("audience")["created"].max().reset_index()
    max_causal = max_causal.merge(models[models["type"] == "causal"], on=["audience", "created"], how="left")
    models = pd.concat([max_conversion, max_causal], ignore_index=True)
    for irow, row in models.iterrows():
        path = row.path
        created = row.created
        if pd.Timestamp.now(tz='UTC') - pd.to_datetime(created, utc=True) > timedelta(days=90):
            print(f"Model created more than 90 days ago, skipping. Path = {path}, created = {created}")
            continue
        print(f"Path = {path}, created = {created}")
        best_model_files = s3.list_files_with_pagination(bucket_name=f"innkeepr-targeting-{sanitized_workspace}", prefix=path)
        path_coding = path.replace("_best_models/", "-coding/").replace("_best_models_lstm/", "-coding_models_lstm/").replace("_best_models_likely/", "-coding_models_likely/")
        best_coding_files = s3.list_files_with_pagination(bucket_name=f"innkeepr-targeting-{sanitized_workspace}", prefix=path_coding)
        cv_best_model = best_model_files[0].split("cv_")[1].split("_")[0]
        expected_standard_scaler = f"StatsStandardScalar_cv_{cv_best_model}.parquet"
        found_file = [file for file in best_coding_files if expected_standard_scaler in file]
        if not found_file:
            if "best_models_lstm" in path:
                path_configs = "configs_models_lstm"
            elif "best_models_likely" in path:
                path_configs = "configs_models_likely"
            else:
                path_configs = "configs"
            path_configs = path.split("/")[0] + "/" + path_configs + "/"
            config_files = s3.list_files_with_pagination(bucket_name=f"innkeepr-targeting-{sanitized_workspace}", prefix=path_configs)
            found_config_file = [file for file in config_files if expected_standard_scaler in file]
            if not found_config_file:
                print(f"Could not find expected standard scaler file: {expected_standard_scaler} in coding files: {best_coding_files} or config files: {config_files} for workspace {workspace['name']} and model path {path}")
                add_scaling_file.append({"workspace": workspace["name"], "audience": row["audience"], "model_path": path, "added_file": None, "error": "No files found"})
                continue
            new_prefix = path_coding.split("/")[0] + "/" + path_coding.split("/")[1] + "/" + found_config_file[0].split("/")[-1]
            print(f"Path configs = {path_configs}")
            print(f"Best model files: {best_model_files}")
            print(f"Best coding files: {best_coding_files}")
            print(f"cv_best_model: {cv_best_model}")
            print(f"Expected standard scaler: {expected_standard_scaler}")
            print(f"Found file: {found_file}")
            print(f"found_config_file = {found_config_file}")
            print(f"new_prefix = {new_prefix}")
            print(f"####### END #########")
            s3.copy_s3_file(
                bucket_name=f"innkeepr-targeting-{sanitized_workspace}",
                prefix=found_config_file[0],
                new_prefix=new_prefix,
            )
            add_scaling_file.append({"workspace": workspace["name"], "audience": row["audience"], "model_path": path, "added_file": new_prefix})
    check_workspaces.append(workspace["name"])
print(set(check_workspaces))
print(add_scaling_file)
with open(f'DataChecks/models/add_scaling_file_{datetime.now()}.json', 'w') as f:
    json.dump(add_scaling_file, f, indent=4)

In [ ]:
with open(f'DataChecks/models/add_scaling_file_{datetime.now()}.json', 'w') as f:
    json.dump(add_scaling_file, f, indent=4)

In [ ]:
error_files = [entry for entry in add_scaling_file if "error" in entry]
error_files_df = pd.DataFrame(error_files)
error_files_df.drop_duplicates(subset=["model_path"])